# AorticTurb — Pass 3 canonical Colab runner

This notebook is a thin orchestration layer over **AorticTurb**. Scientific calculations remain in the tested package.

Pass 3 executes only the agreed sequence: verified staging/startup gate → fresh phase-conditioned decomposition → Gate A with the unchanged legacy primary support rule → the pre-specified support-policy sensitivity analysis.

It does not call the older `run_all` workflow, does not promote outputs to canonical `05_results`, and does not automatically enter downstream KO/Taylor/Lumley/diagnostics/classification stages.

Locked Pass-3 runner/package commit: `f48d6564fa29231a432543f721ef0448f719c3b9`

Persistent outputs are written under `<Google Drive project root>/results/`; canonical raw Drive data are never modified.


In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

AORTICTURB_REPO = "https://github.com/khalid-saqr/AorticTurb.git"
AORTICTURB_COMMIT = "f48d6564fa29231a432543f721ef0448f719c3b9"
EXPECTED_AORTICTURB_VERSION = "0.2.0"
WORKTREE = Path("/content/AorticTurb")

if sys.version_info[:2] not in {(3, 10), (3, 11), (3, 12), (3, 13)}:
    raise RuntimeError(f"UNSUPPORTED_PYTHON: {sys.version.split()[0]}")
if WORKTREE.exists():
    if not str(WORKTREE).startswith("/content/"):
        raise RuntimeError(f"REFUSING_TO_REMOVE_NON_SCRATCH_PATH: {WORKTREE}")
    shutil.rmtree(WORKTREE)

subprocess.run(["git", "clone", "--quiet", AORTICTURB_REPO, str(WORKTREE)], check=True)
subprocess.run(["git", "-C", str(WORKTREE), "checkout", "--quiet", "--detach", AORTICTURB_COMMIT], check=True)
observed_commit = subprocess.run(["git", "-C", str(WORKTREE), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
if observed_commit != AORTICTURB_COMMIT:
    raise RuntimeError(f"GIT_COMMIT_MISMATCH: expected {AORTICTURB_COMMIT}, got {observed_commit}")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", str(WORKTREE)], check=True)
os.chdir(WORKTREE)

import aorticturb
if aorticturb.__version__ != EXPECTED_AORTICTURB_VERSION:
    raise RuntimeError(f"INSTALL_FAILED: expected {EXPECTED_AORTICTURB_VERSION}, got {aorticturb.__version__}")

runner_path = WORKTREE / "notebooks" / "pass3_runner.py"
spec = importlib.util.spec_from_file_location("aorticturb_pass3_runner", runner_path)
runner = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(runner)

print("AorticTurb", aorticturb.__version__)
print("Pass-3 runner/package commit", observed_commit)


In [ ]:
from IPython.display import display
import pandas as pd

# EDIT ONLY THIS PATH IF THE PROJECT IS MOUNTED ELSEWHERE.
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study"

pass3 = runner.run_pass3(
    expected_commit=AORTICTURB_COMMIT,
    drive_project_root=DRIVE_PROJECT_ROOT,
)

display(pd.DataFrame(pass3["support_table"]))
display(pd.DataFrame(pass3["sensitivity_table"]))
print("Primary Gate A:", pass3["primary_gate_a"])
print("Pass-3 record:", pass3["record_dir"])
print("Canonical 05_results promoted:", pass3["canonical_05_results_promoted"])
